# 01 — 決定木と CART：分割は何を最大化するか
## 予測を区分的定数関数として考える
回帰木の葉 $R_j$ は定数 $c_j$ を返す。

$$\arg\min_c\sum_{i:x_i\in R_j}(y_i-c)^2=\frac{1}{|R_j|}\sum_{i:x_i\in R_j}y_i.$$

分割前の $\mathrm{SSE}_{parent}$ と分割後の $\mathrm{SSE}_L+\mathrm{SSE}_R$ の差

$$\mathrm{gain}(t)=\mathrm{SSE}_{parent}-\mathrm{SSE}_L(t)-\mathrm{SSE}_R(t)$$

を最大化する閾値 $t$ を選ぶ。各側に少なくとも \`min_samples_leaf\` 個必要。全特徴量を扱う CART の簡略版として、まず **1 特徴量／深さ 1（stump）** を実装する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree
X = np.arange(8.).reshape(-1,1)
y = np.array([1,1,2,2,7,7,8,8.])
tree = DecisionTreeRegressor(max_depth=1, random_state=42).fit(X,y)
fig, axes = plt.subplots(1, 2, figsize=(11,3))
axes[0].scatter(X[:,0],y,label="observations")
xx = np.linspace(0,7,200).reshape(-1,1)
axes[0].plot(xx,tree.predict(xx),label="one split")
axes[0].legend(); axes[0].set_title("Piecewise-constant prediction")
plot_tree(tree,feature_names=["x"],ax=axes[1],filled=False)
plt.tight_layout(); plt.show()

## 計算例
$x=(0,1,2,3)$, $y=(0,0,10,10)$。分割前は平均5で SSE = $4\times25=100$。閾値1.5なら左右の SSE は0。従って gain = **100**。

候補の閾値は、ソート後の異なる値の **中点**（例: 0.5, 1.5, 2.5）。同点なら最小の閾値を返す仕様にする。分割による改善がない場合、(None, 0) とし、葉を一つにする。

In [ ]:
x = np.array([0.,1.,2.,3.]); y = np.array([0.,0.,10.,10.])
for t in [0.5,1.5,2.5]:
    l = y[x <= t]; r = y[x > t]
    gain = np.sum((y-y.mean())**2) - np.sum((l-l.mean())**2) - np.sum((r-r.mean())**2)
    print(f"threshold={t:.1f}, gain={gain:.1f}")

## 演習（3関数）
1. \`best_split_1d\`: 候補列挙、gain 計算、最良選択。
2. \`fit_stump\`: 最適閾値・左右の平均を dict に格納。
3. \`predict_stump\`: 新しい x を左右へ分類し予測。

**思考問題:** 外れ値が1点だけあると SSE はどう反応する？ 決定木の深さを無制限にすると、訓練誤差と汎化誤差はどうなる？

In [ ]:
print('採点コマンド: python tools/grade.py -k "test_best_split or test_stump"')

**次章:** 強い1本の木ではなく、少し異なる木を多数学習して平均する。